# 01 — BERT-base Baseline (SST-2)

**Branch:** `reproduction/bert-baseline`  
**Owner:** Person B

Fine-tunes `bert-base-uncased` on SST-2 and measures accuracy, model size, and inference speed.
These numbers serve as the teacher/baseline for all subsequent comparisons.

**Expected results (from paper):** ~93.5% accuracy on SST-2 validation set.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
import sys, os
sys.path.append('../src')

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

from train import train_model
from evaluate import evaluate_model
from utils import print_model_info, measure_inference_speed, save_results

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MODEL_NAME   = 'bert-base-uncased'
MAX_LENGTH   = 128
BATCH_SIZE   = 32
EPOCHS       = 3
LR           = 2e-5
SEED         = 42

torch.manual_seed(SEED)

In [ ]:
# ── Load & tokenize SST-2 ──────────────────────────────────────────────────
raw = load_dataset('glue', 'sst2')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=MAX_LENGTH, padding='max_length')

tokenized = raw.map(tokenize, batched=True)
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'label'])
tokenized = tokenized.rename_column('label', 'labels')

train_loader = DataLoader(tokenized['train'],    batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(tokenized['validation'], batch_size=BATCH_SIZE)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print_model_info(model, 'BERT-base')

In [ ]:
# ── Fine-tune ──────────────────────────────────────────────────────────────
history = train_model(model, train_loader, val_loader, DEVICE, epochs=EPOCHS, lr=LR)
print('Training complete.')

In [ ]:
# ── Evaluate ───────────────────────────────────────────────────────────────
eval_results = evaluate_model(model, val_loader, DEVICE)
print(f"Validation accuracy: {eval_results['accuracy']}%")

In [ ]:
# ── Benchmark inference speed ──────────────────────────────────────────────
speed = measure_inference_speed(model, val_loader, DEVICE)
print(speed)

In [ ]:
# ── Save results ───────────────────────────────────────────────────────────
from utils import count_parameters, model_size_mb

total_params, _ = count_parameters(model)

results = {
    'model':             MODEL_NAME,
    'task':              'sst2',
    'val_accuracy':      eval_results['accuracy'],
    'total_parameters':  total_params,
    'model_size_mb':     model_size_mb(model),
    'inference_speed':   speed,
    'training_history':  history,
}

save_results(results, 'bert_baseline.json')
print(results)